<a href="https://colab.research.google.com/github/fc63/gender-classification/blob/main/gpmodel_v3/1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets==3.6.0 transformers torch evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 91.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import warnings
import evaluate
import pandas as pd
import numpy as np
import os
import re
import pickle
import gc
import torch
import shutil
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback, TrainerCallback
from tqdm import tqdm
from google.colab import drive
from transformers import EarlyStoppingCallback

drive.mount('/content/drive')

with open("/content/drive/MyDrive/datasets/merged_europarl_informal.pkl", "rb") as f:
    df = pickle.load(f)

Mounted at /content/drive


In [ ]:
# label encode: female -> 0, male -> 1
label_map = {'female': 0, 'male': 1}
df['label'] = df['gender'].map(label_map)
print(df['label'].value_counts())

label
0    591027
1    591027
Name: count, dtype: int64


In [ ]:
# early stopping by overriding eval_f1 and eval_loss metrics to prevent over fitting
class DualMetricEarlyStoppingCallback(TrainerCallback):
    def __init__(self, patience=3, min_delta_f1=1e-7, min_delta_loss=1e-7, tolerance_f1=0.02, tolerance_loss=0.02):
        self.patience = patience
        self.counter = 0
        self.best_f1 = None
        self.best_loss = None
        self.min_delta_f1 = min_delta_f1
        self.min_delta_loss = min_delta_loss
        self.tolerance_f1 = tolerance_f1
        self.tolerance_loss = tolerance_loss

    def on_evaluate(self, args, state, control, metrics, **kwargs):
        current_f1 = metrics.get("eval_f1")
        current_loss = metrics.get("eval_loss")

        if current_f1 is None or current_loss is None:
            return control

        if self.best_f1 is None or self.best_loss is None:
            self.best_f1 = current_f1
            self.best_loss = current_loss
            self.counter = 0
        else:
            f1_improved = current_f1 > self.best_f1 + self.min_delta_f1
            f1_decline_ok = current_f1 >= self.best_f1 - self.tolerance_f1

            loss_improved = current_loss < self.best_loss - self.min_delta_loss
            loss_decline_ok = current_loss <= self.best_loss + self.tolerance_loss

            if f1_improved or loss_improved or (f1_decline_ok and loss_decline_ok):
                self.best_f1 = max(self.best_f1, current_f1)
                self.best_loss = min(self.best_loss, current_loss)
                self.counter = 0
            else:
                self.counter += 1
                print(f"[EarlyStopping] No acceptable improvement. Patience {self.counter}/{self.patience}")

        if self.counter >= self.patience:
            print(f"[EarlyStopping] Triggered at epoch {state.epoch}.")
            control.should_training_stop = True

        return control

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if not isinstance(logits, torch.Tensor):
        logits = torch.tensor(logits)
    if not isinstance(labels, torch.Tensor):
        labels = torch.tensor(labels)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    preds = logits.argmax(dim=1).to(device)
    labels = labels.to(device)

    num_classes = torch.max(labels).item() + 1
    f1_total = 0.0
    precision_total = 0.0
    recall_total = 0.0

    for cls in range(num_classes):
        tp = ((preds == cls) & (labels == cls)).sum()
        fp = ((preds == cls) & (labels != cls)).sum()
        fn = ((preds != cls) & (labels == cls)).sum()

        precision = tp / (tp + fp + 1e-8)
        recall = tp / (tp + fn + 1e-8)
        f1 = 2 * precision * recall / (precision + recall + 1e-8)

        precision_total += precision
        recall_total += recall
        f1_total += f1

    macro_precision = precision_total / num_classes
    macro_recall = recall_total / num_classes
    macro_f1 = f1_total / num_classes

    accuracy = (preds == labels).sum().float() / labels.shape[0]

    return {
        "f1": macro_f1.item(),
        "precision": macro_precision.item(),
        "recall": macro_recall.item(),
        "accuracy": accuracy.item()
    }

# which model saved
class PrintBestModelCallback(TrainerCallback):
    def on_train_end(self, args, state, control, **kwargs):
        best_step = getattr(state, "best_step", None)
        best_metric = getattr(state, "best_metric", None)

        if best_step is not None and best_metric is not None:
            print(f"\n[INFO] Best model was at step {best_step} with best {args.metric_for_best_model}: {best_metric}")
        else:
            print("\n[INFO] Best step or metric not available (possibly due to resumed checkpoint).")


# split dataset
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['label'], test_size=0.12, random_state=63, stratify=df['label'])

# tokenization for huggingfaceapi
train_df = {"text": list(X_train), "label": list(y_train)}
test_df = {"text": list(X_test), "label": list(y_test)}
train_dataset = Dataset.from_dict(train_df)
eval_dataset = Dataset.from_dict(test_df)
model_path = "/content/drive/MyDrive/models/gp_model_24750"
tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False)
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )
train_dataset = train_dataset.map(tokenize_function, batched=True)
eval_dataset = eval_dataset.map(tokenize_function, batched=True)
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
eval_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

#load model
model = AutoModelForSequenceClassification.from_pretrained(model_path)

stopsteps = 250
train_size = len(train_dataset)
batch_size = 64
epochs = 2

max_steps = (train_size // batch_size) * epochs
warmup_steps = int(0.05 * max_steps)

# define training arguments
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/models/gp_model_fulldatasetfinetuned",
    eval_strategy="steps",
    eval_steps=stopsteps,
    save_strategy="steps",
    save_steps=stopsteps,
    learning_rate=1e-6,
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=epochs,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=stopsteps,
    save_total_limit=None,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    gradient_accumulation_steps=1,
    fp16=True,
)

# define trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks = [DualMetricEarlyStoppingCallback(
    patience=14,
    min_delta_f1=1e-7,
    min_delta_loss=1e-7,
    tolerance_f1=0.05,
    tolerance_loss=0.03
    ),
                 PrintBestModelCallback()]
)

# training model
trainer.train()

# saving model
save_path = "/content/drive/MyDrive/models/gp_modelv3"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

Map:   0%|          | 0/1040207 [00:00<?, ? examples/s]

Map:   0%|          | 0/141847 [00:00<?, ? examples/s]

<ipython-input-4-2b0b53a72824>:158: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss,Validation Loss,F1,Precision,Recall,Accuracy
250,0.659500,0.639507,0.631663,0.631672,0.631667,0.631667
500,0.637100,0.625659,0.646026,0.646422,0.646175,0.646175
750,0.631600,0.619628,0.650437,0.652439,0.651167,0.651166
1000,0.630100,0.616208,0.655086,0.655958,0.655397,0.655396
1250,0.622700,0.613564,0.657046,0.658210,0.657455,0.657455
1500,0.616700,0.611110,0.658776,0.659051,0.658872,0.658872
1750,0.619300,0.611078,0.657391,0.662437,0.659133,0.659133
2000,0.614000,0.608530,0.660792,0.662408,0.661347,0.661346
2250,0.619600,0.607407,0.662931,0.663287,0.663053,0.663052
2500,0.612000,0.607659,0.658564,0.665456,0.660910,0.660909



[INFO] Best step or metric not available (possibly due to resumed checkpoint).


('/content/drive/MyDrive/models/gp_modelv3/tokenizer_config.json',
 '/content/drive/MyDrive/models/gp_modelv3/special_tokens_map.json',
 '/content/drive/MyDrive/models/gp_modelv3/spm.model',
 '/content/drive/MyDrive/models/gp_modelv3/added_tokens.json')

In [ ]:
# eval loglarını filtrele
eval_logs = [log for log in trainer.state.log_history if "eval_loss" in log]

# dataframe'e çevir
eval_df = pd.DataFrame(eval_logs)

# tarih veya adım bilgisiyle adlandırmak istersen
log_save_path = "/content/drive/MyDrive/models/training_logs_eval_v3mdoel.csv"

# csv olarak kaydet
eval_df.to_csv(log_save_path, index=False)

print(f"[INFO] Eval logları kaydedildi: {log_save_path}")

[INFO] Eval logları kaydedildi: /content/drive/MyDrive/models/training_logs_eval_v3mdoel.csv


In [ ]:
import requests

def send_telegram_message(message):
    token = "7791020893:AAGIXZbLRG6YVNXaNhRkCNiQDUV2-jXsDJY"
    chat_id = 7689600055
    url = f"https://api.telegram.org/bot{token}/sendMessage"
    data = {"chat_id": chat_id, "text": message}
    requests.post(url, data=data)

send_telegram_message("Colab çalışman tamamlandı 🎉 Yeniden finetune için.......")